# 最後 7.32% 複核與 Backtest Eligibility v0.1（唯讀、離線）

依序跑第 1–5 格，全程不發網路請求、不呼叫富邦、不改寫任何 raw、manifest 或 pointer。

- **第 4 格**分三個階段執行：
  - 階段 1：重現既有對帳數字。對不上就停止，只輸出 `phase1_reproduce.json`。
  - 階段 2：逐一複核未驗證的「交易所×日期」。項目包括：成分報表完整度、原始檔 SHA-256、以現行解析器重新解析原始 gzip、殘差結構，並依規則 v2 重新分類。
  - 階段 3：輸出 `exchange_date_quality.parquet`、`symbol_day_quality.parquet`、`backtest_eligibility_manifest.json`，以及 2026 年成交量短少報表、分類差異表、未解決清單、Level A / B 統計。
- **第 5 格**把結果複製到 Drive 的 `audits/eligibility_v0.1/`。

預計 10–30 分鐘，大部分時間花在從 Drive 讀取 832 個參考資料 shard。

In [ ]:
# 1. 參數
REPO_OWNER, REPO_NAME, BRANCH = "hh4832", "tw-market-data-lake", "main"
DATA_ROOT = "/content/drive/MyDrive/00Quant_Research/data"
LEDGER_DIR = f"{DATA_ROOT}/_tw-market-data-lake/audits/gap_ledger_20261026_163358_3e8eed0"
RECON_DIR = f"{DATA_ROOT}/_tw-market-data-lake/audits/official_volume_batch_recon/volume_recon_20261027_181218"
DATASET_ROOT = f"{DATA_ROOT}/_tw-market-data-lake/reference/official_volume/v1"
OUT_DIR = "/content/tw_lake_eligibility"
PROJECT_DIR = f"/content/{REPO_NAME}"

In [ ]:
# 2. 掛載 Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 3. clone / pull + 依賴
import subprocess, os
if os.path.exists(f"{PROJECT_DIR}/.git"):
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git", PROJECT_DIR], check=True)
print("HEAD:", subprocess.check_output(["git", "-C", PROJECT_DIR, "rev-parse", "HEAD"], text=True).strip())
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml>=6", "tqdm>=4.66"], check=True)
def stream(args):
    env = dict(os.environ, PYTHONPATH=f"{PROJECT_DIR}/src", PYTHONUNBUFFERED="1")
    p = subprocess.Popen(args, cwd=PROJECT_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    return p.wait()

def ensure_drive():
    """Mount Drive safely. If an earlier cell unmounted Drive and then wrote under /content/drive,
    those files are only on this runtime's disk: move them aside (never delete) and copy any audit
    folders that are missing on the real Drive."""
    import os, shutil, time
    from google.colab import drive
    if os.path.ismount("/content/drive"):
        return
    stray = None
    if os.path.isdir("/content/drive") and os.listdir("/content/drive"):
        stray = f"/content/drive_local_{int(time.time())}"
        shutil.move("/content/drive", stray)
        print("moved local (unsynced) files to", stray)
    drive.mount("/content/drive")
    if stray:
        src_root = f"{stray}/MyDrive/00Quant_Research/data/_tw-market-data-lake/audits"
        dst_root = f"{DATA_ROOT}/_tw-market-data-lake/audits"
        for dirpath, dirnames, filenames in os.walk(src_root):
            rel = os.path.relpath(dirpath, src_root)
            for d in list(dirnames):
                target = os.path.join(dst_root, rel, d)
                if not os.path.exists(target):
                    shutil.copytree(os.path.join(dirpath, d), target)
                    print("copied missing", target)
                    dirnames.remove(d)

In [ ]:
# 4. 執行（唯讀、離線）
import glob
print("exit", stream([sys.executable, "-m", "tw_lake.audit.eligibility", "--data-root", DATA_ROOT,
                      "--ledger-dir", LEDGER_DIR, "--recon-dir", RECON_DIR, "--dataset-root", DATASET_ROOT,
                      "--out-dir", OUT_DIR]))
ELIG_DIR = sorted(glob.glob(f"{OUT_DIR}/eligibility_*"))[-1]
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_rows", 200)
for f in ("unresolved_exchange_dates.csv", "refetch_minimal.csv"):
    try:
        t = pd.read_csv(f"{ELIG_DIR}/{f}")
        print(f"\n== {f} ({len(t)} rows)"); print(t.head(60).to_string(index=False))
    except FileNotFoundError:
        pass

In [ ]:
# 5. 複製到 Drive
import shutil, os
dest = f"{DATA_ROOT}/_tw-market-data-lake/audits/eligibility_v0.1/{os.path.basename(ELIG_DIR)}"
shutil.copytree(ELIG_DIR, dest)
print("copied to", dest)
from google.colab import drive
drive.flush_and_unmount()